In [1]:
import sys, json, warnings
sys.path.insert(0, "..")            # ml-service root, so `training` and `app` import
warnings.filterwarnings("ignore")
import duckdb, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from training import config as C
sns.set_theme(style="whitegrid")
IMG = C.ROOT_DIR / "docs" / "images"; IMG.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(); con.execute("SET memory_limit='1500MB'"); con.execute("SET threads=2")
G = str(C.FLIGHTS_GLOB).replace("\\", "/")
def q(sql): return con.execute(sql.replace("{G}", G)).df()
def save(name):
    plt.tight_layout(); plt.savefig(IMG / f"{name}.png", dpi=110); plt.show()
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score
from training.feature_engineering import load_xy

# 05 - XGBoost tuning (delay classifier)
Small random search on a 15% train sample, scored on September (validation). Test months are never touched.

In [2]:
Xtr, ytr = load_xy("train", "delay", sample=0.15); Xva, yva = load_xy("valid", "delay", sample=0.5)
spw = (ytr == 0).sum() / (ytr == 1).sum(); rng = np.random.default_rng(0); rows = []
for i in range(8):
    p = dict(max_depth=int(rng.choice([4, 5, 6, 7, 8])), learning_rate=float(rng.choice([0.05, 0.08, 0.1])),
             min_child_weight=int(rng.choice([5, 10, 30, 60])), subsample=float(rng.choice([0.7, 0.8, 0.9])),
             colsample_bytree=float(rng.choice([0.6, 0.8, 1.0])))
    m = xgb.XGBClassifier(n_estimators=400, tree_method="hist", n_jobs=3, scale_pos_weight=spw, eval_metric="aucpr",
                          early_stopping_rounds=25, random_state=42, **p).fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    pr = m.predict_proba(Xva)[:, 1]
    rows.append({**p, "trees": m.best_iteration, "val_auc": roc_auc_score(yva, pr), "val_pr_auc": average_precision_score(yva, pr)})
T = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).round(4); T

,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,trees,val_auc,val_pr_auc
3,7,0.08,30,0.8,1.0,167,0.6908,0.3570
7,6,0.05,10,0.8,0.8,195,0.6899,0.3553
6,8,0.05,5,0.9,0.6,143,0.6905,0.3552
2,7,0.10,30,0.8,1.0,96,0.6904,0.3547
0,8,0.08,30,0.7,0.6,96,0.6892,0.3532
1,4,0.05,5,0.7,1.0,392,0.6895,0.3532
5,8,0.08,5,0.9,1.0,119,0.6882,0.3522
4,5,0.10,30,0.7,0.8,80,0.6866,0.3511


The production parameters in `training/train_xgboost.py` (depth 7, lr 0.1, mcw 10, subsample/colsample 0.8) sit in the plateau of this search: differences between reasonable settings are small, the limit is feature information, not hyperparameters.